# ER Diagram — fvr Data Model (PK & FK)

**System:** File Validation & Routing Framework (`fvr`, MongoDB) • **Version:** 1.0 • **Status:** DRAFT FOR REVIEW

## 1. Purpose & Notation

Entity-relationship view of the 21 collections + GridFS (`fs.files`). **Notation is relational, enforcement is not** — MongoDB has no declarative PK/FK constraints:

* **PK** = the collection's prefixed `_id` string — uniqueness enforced by the default `_id` index (or explicit unique index where noted).
* **UK** = additional unique key — enforced by explicit **unique indexes** (e.g. `file_registry.sha256_checksum`, `event_log (event_type, dedup_key)`).
* **FK** = application-enforced reference — the FastAPI write layer validates existence at write time; reconciliation (Phase 3) audits integrity after the fact.

Companion docs: `Database Design LLD` (field-level specs — the source of truth for this diagram), `Database Design HLD` (architecture & decisions).

## 2. Layered Overview

```
REGISTRY (config, versioned)          EVENTING (queue)
┌───────────┐ ┌──────────────┐       ┌──────────┐
│ users      │ │ source_      │       │ event_log │
└─────┬─────┘ │ systems      │       └────┬─────┘
      │       └──────┬───────┘            │ correlation_id ties
┌─────┴──────┐       │                    │ every artifact
│ file_types │◄──────┼─────┐              │ together
└─┬───┬───┬──┘       │     │              │
  │   │   │      ┌────▼─────┴───┐          │
  │   │   └────►│ event-driven │          │
  │   │         │ adapters     │          │
  │   │         └──────────────┘          │
  │   │                                   │
┌─▼───┴───┐ ┌────────────┐ ┌───────────┐  │
│ schema_ │ │ validation │ │ routing_  │  │
│ versions│ │ rules      │ │ policies  │  │
└────┬────┘ └─────┬──────┘ └─────┬─────┘  │
     │            │              │        │
EXECUTION (runtime state)        │        │
┌────▼───────────────────────────▼────┐  │
│ upload_sessions ──► file_registry ◄──┼──┘
│ (users, ip, machine)   │ state machine │
│        ┌───────────────┼───────────┐  │
│        │               │           │  │
│ validation_results  file_transfers │  │
│ (per rule × file)    (routing manifest, checksum) │
│ batches (Ph2)                        │  │
└─────────────────────────────────────┘  │
GOVERNANCE: audit_log (append-only) • file_lifecycle • notifications • archive_jobs
RESILIENCE (Ph3): databricks_ingestion_results • reconciliation_runs • reconciliation_results
STORAGE: GridFS fs.files/fs.chunks (bytes only — file_registry.gridfs_file_id → fs.files._id)
```

## 3. ER Diagram (Mermaid source)

Paste into [mermaid.live](https://mermaid.live), or save as a `.md` file on GitHub (GitHub renders Mermaid natively).

```mermaid
erDiagram
    USERS ||--o{ UPLOAD_SESSIONS : starts
    SOURCE_SYSTEMS ||--o{ UPLOAD_SESSIONS : receives
    SOURCE_SYSTEMS ||--o{ FILE_REGISTRY : delivers
    SOURCE_SYSTEMS ||--o{ EVENT_LOG : produces
    UPLOAD_SESSIONS ||--o{ FILE_REGISTRY : contains
    FILE_TYPES ||--o{ FILE_REGISTRY : classifies
    FILE_TYPES ||--o{ SCHEMA_VERSIONS : versions
    FILE_TYPES ||--o{ VALIDATION_RULES : binds
    FILE_TYPES ||--o{ VOLUME_MAP : maps
    FILE_TYPES ||--o{ RETENTION_POLICIES : scopes
    SCHEMA_VERSIONS ||--o{ FILE_REGISTRY : validates
    FILE_REGISTRY }|--|| FS_FILES : gridfs_file_id
    FILE_REGISTRY ||--o{ VALIDATION_RESULTS : yields
    VALIDATION_RULES ||--o{ VALIDATION_RESULTS : produces
    ROUTING_POLICIES ||--o{ FILE_TRANSFERS : decides
    FILE_REGISTRY ||--o{ FILE_TRANSFERS : moved-by
    BATCHES ||--o{ FILE_REGISTRY : groups
    FILE_REGISTRY ||--o{ FILE_LIFECYCLE : traced-by
    RETENTION_POLICIES ||--o{ ARCHIVE_JOBS : drives
    ARCHIVE_JOBS ||--o{ FILE_LIFECYCLE : writes
    FILE_REGISTRY ||--o{ DATABRICKS_INGESTION_RESULTS : reported-for
    FILE_REGISTRY ||--o{ RECONCILIATION_RESULTS : certified-by
    RECONCILIATION_RUNS ||--o{ RECONCILIATION_RESULTS : contains

    USERS { string user_id PK
            string email UK
            array roles
            bool is_active }
    SOURCE_SYSTEMS { string source_id PK
                      string source_type
                      string connection_ref
                      bool is_active }
    FILE_TYPES { string file_type_id PK
                 string code UK
                 array allowed_extensions
                 bool is_active }
    SCHEMA_VERSIONS { string schema_id PK
                      string file_type_id FK
                      int version
                      string status
                      date effective_from
                      array fields }
    VALIDATION_RULES { string rule_id PK
                       string rule_code
                       string category
                       string severity
                       string applies_to FK
                       object params
                       int priority
                       int version }
    ROUTING_POLICIES { string policy_id PK
                       int priority
                       object match
                       string action }
    VOLUME_MAP { string map_id PK
                 string file_type_id FK
                 string destination_type
                 string volume_root
                 string subfolder
                 string naming_convention }
    RETENTION_POLICIES { string policy_id PK
                         string file_type_id FK
                         string status_class
                         int retention_days
                         int archive_after_days
                         int purge_after_days
                         bool legal_hold }
    EVENT_LOG { string event_id PK
                string event_type
                string source_id FK
                string dedup_key
                string detection_method
                string processing_status
                string correlation_id }
    UPLOAD_SESSIONS { string session_id PK
                      string source_id FK
                      string user_id FK
                      string client_ip
                      string machine_name
                      int files_declared_count
                      int files_received_count
                      array parts }
    FILE_REGISTRY { string file_id PK
                    string source_id FK
                    string upload_session_id FK
                    string file_type_id FK
                    string schema_version_id FK
                    objectId gridfs_file_id FK
                    string sha256_checksum UK
                    string current_status
                    string batch_id FK
                    date received_at }
    VALIDATION_RESULTS { string result_id PK
                          string file_id FK
                          string rule_id FK
                          int rule_version
                          string status
                          object details }
    BATCHES { string batch_id PK
              string source_id FK
              string triggered_by
              object watermark
              string status }
    FILE_TRANSFERS { string transfer_id PK
                     string file_id FK
                     string policy_id FK
                     string direction
                     string status
                     string source_sha256
                     string dest_sha256 }
    FILE_LIFECYCLE { string event_id PK
                     string file_id FK
                     string location
                     date timestamp
                     string triggered_by_job FK }
    NOTIFICATIONS { string notification_id PK
                    string event
                    object condition
                    string channel
                    bool is_active }
    ARCHIVE_JOBS { string job_id PK
                   string policy_id FK
                   string status
                   int files_scanned
                   int files_purged }
    DATABRICKS_INGESTION_RESULTS { string ingestion_id PK
                                    string file_id FK
                                    string run_id
                                    string bronze_table
                                    int rows_read
                                    int rows_written }
    RECONCILIATION_RUNS { string run_id PK
                          string scope
                          int expected_files
                          int matched
                          int discrepancies }
    RECONCILIATION_RESULTS { string result_id PK
                             string file_id FK
                             string run_id FK
                             string status
                             int row_delta
                             string severity }
    FS_FILES { objectId _id PK
               string filename
               int length
               date uploadDate
               object metadata }
```

> `NOTIFICATIONS` has no FKs (stand-alone config). `VOLUME_MAP.file_type_id` / `RETENTION_POLICIES.file_type_id` accept the wildcard `*` (all file types) — see cell 5.

## 4. PK / UK / FK Reference Table

| Collection | PK | Unique keys (UK) | Foreign keys (FK → target) |
| --- | --- | --- | --- |
| `users` | `user_id` | `email` | — |
| `source_systems` | `source_id` | — | — |
| `file_types` | `file_type_id` | `code` | — |
| `schema_versions` | `schema_id` | `(file_type_id, version)` | `file_type_id → file_types` |
| `validation_rules` | `rule_id` | `(rule_code, version)` | `applies_to → file_types` (nullable = global) |
| `routing_policies` | `policy_id` | — | — |
| `volume_map` | `map_id` | — | `file_type_id → file_types` or `*` |
| `retention_policies` | `policy_id` | — | `file_type_id → file_types` or `*` |
| `event_log` | `event_id` | `(event_type, dedup_key)` | `source_id → source_systems` |
| `upload_sessions` | `session_id` | — | `source_id → source_systems`; `user_id → users` |
| `file_registry` | `file_id` | `sha256_checksum` | `source_id → source_systems`; `upload_session_id → upload_sessions`; `file_type_id → file_types`; `schema_version_id → schema_versions`; `gridfs_file_id → fs.files._id`; `batch_id → batches` (nullable, Ph2) |
| `validation_results` | `result_id` | — | `file_id → file_registry`; `rule_id → validation_rules` |
| `batches` | `batch_id` | — | `source_id → source_systems` |
| `file_transfers` | `transfer_id` | partial `(file_id, direction)` where COMPLETED | `file_id → file_registry`; `policy_id → routing_policies` |
| `audit_log` | `event_id` | — | `entity_id → <entity_type>` (polymorphic); `correlation_id → event_log` (nullable) |
| `file_lifecycle` | `event_id` | — | `file_id → file_registry`; `triggered_by_job → archive_jobs` (nullable) |
| `notifications` | `notification_id` | — | — |
| `archive_jobs` | `job_id` | — | `policy_id → retention_policies` |
| `databricks_ingestion_results` | `ingestion_id` | `ingestion_id` (idempotent upsert) | `file_id → file_registry`; `file_type_id → file_types` |
| `reconciliation_runs` | `run_id` | — | — |
| `reconciliation_results` | `result_id` | `(file_id, run_id)` | `file_id → file_registry`; `run_id → reconciliation_runs` |
| `fs.files` (GridFS) | `_id` (ObjectId) | `(filename, uploadDate)` (default) | — (referenced BY `file_registry`) |
| `fs.chunks` (GridFS) | `(files_id, n)` | — | `files_id → fs.files._id` (driver-managed) |

## 5. Cardinality & Integrity Notes

| Relationship | Cardinality | Notes |
| --- | --- | --- |
| users → upload_sessions | 1 : N | one session per portal interaction |
| upload_sessions → file_registry | 1 : N | multi-file upload; completeness via declared vs received counts |
| file_types → schema_versions | 1 : N | **exactly one `active` version at a time** (app-enforced) |
| file_types → validation_rules | 1 : N | via `applies_to`; `null` = global rule |
| file_registry → fs.files | N : 1 | each file has exactly one GridFS object (bytes) |
| file_registry → validation_results | 1 : N | one per executed rule (≈ 8–10 per file in Phase 1) |
| routing_policies → file_transfers | 1 : N | first-match-by-priority decision recorded per transfer |
| file_registry → file_transfers | 1 : N | partial-unique COMPLETED guard = never transferred twice per direction |
| retention_policies → archive_jobs → file_lifecycle | 1 : N, 1 : N | purge trail |
| reconciliation_runs → reconciliation_results | 1 : N | unique `(file_id, run_id)` — one verdict per file per run |
| event_log.correlation_id | 1 : N (logical) | ties results/transfers/lifecycle/audit back to the originating event |

**Integrity model (3 layers):** 1) `$jsonSchema` validators (shape + enums at write time) → 2) FastAPI referential checks (FK targets must exist) → 3) Phase-3 reconciliation (audits the chain end-to-end; e.g. `MISSING_IN_BRONZE` is an integrity verdict, not just a data gap).